In [2]:
!pip install scipy scikit-learn matplotlib pandas numpy

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import json
import numpy as np
import pickle
from pathlib import Path
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# ============================================================
# 1. LOAD DATA
# ============================================================

def load_hashtag_data(file_path: str) -> dict:
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

# ============================================================
# 2. EMBEDDING
# ============================================================

def embed_texts_batch(texts, model, batch_size=64):
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        print(f"Embedding batch {i//batch_size + 1}")

        embeddings = model.encode(
            batch,
            batch_size=batch_size,
            normalize_embeddings=True,
            show_progress_bar=False
        )

        all_embeddings.append(embeddings)

    return np.vstack(all_embeddings)


def create_and_save_embeddings(data, output_path, batch_size=32):
    print("Loading model...")
    model = SentenceTransformer('BAAI/bge-m3')

    hashtags = list(data.keys())
    texts = list(data.values())

    embeddings = embed_texts_batch(texts, model, batch_size)

    result = {
        'hashtags': hashtags,
        'texts': texts,
        'embeddings': embeddings,
        'mapping': data
    }

    with open(output_path, 'wb') as f:
        pickle.dump(result, f)

    print("Saved embeddings")
    return result


def load_embeddings(file_path):
    with open(file_path, 'rb') as f:
        return pickle.load(f)

# ============================================================
# 3. KMEANS CLUSTERING
# ============================================================

def run_kmeans_clustering(embeddings, n_clusters_list):
    models = {}
    cluster_results = {}

    for n_clusters in n_clusters_list:
        print(f"Running KMeans with {n_clusters} clusters...")

        kmeans = KMeans(
            n_clusters=n_clusters,
            random_state=42,
            n_init=10
        )

        labels = kmeans.fit_predict(embeddings)

        models[n_clusters] = kmeans
        cluster_results[n_clusters] = labels

    return models, cluster_results


def save_cluster_results(hashtags, texts, cluster_results, output_dir):
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True)

    for n_clusters, labels in cluster_results.items():
        df = pd.DataFrame({
            'hashtag': hashtags,
            'text': texts,
            'cluster': labels
        })

        df = df.sort_values('cluster')

        df.to_csv(output_path / f'clusters_{n_clusters}.csv', index=False)

        with open(output_path / f'clusters_{n_clusters}_summary.txt', 'w', encoding='utf-8') as f:
            for cluster_id in sorted(df['cluster'].unique()):
                cluster_items = df[df['cluster'] == cluster_id]

                f.write(f"\n--- Cluster {cluster_id} ({len(cluster_items)}) ---\n")

                for _, row in cluster_items.iterrows():
                    f.write(f"{row['hashtag']}: {row['text']}\n")

# ============================================================
# 4. PCA VISUALIZATION
# ============================================================

def visualize_clusters_pca(embeddings, cluster_results, texts, output_dir):
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True)

    pca = PCA(n_components=2)
    emb_2d = pca.fit_transform(embeddings)

    for n_clusters, labels in cluster_results.items():
        plt.figure(figsize=(12, 8))

        plt.scatter(
            emb_2d[:, 0],
            emb_2d[:, 1],
            c=labels,
            cmap='tab20',
            alpha=0.7
        )

        plt.title(f"KMeans - {n_clusters} clusters")

        plt.savefig(output_path / f'pca_{n_clusters}.png')
        plt.close()

# ============================================================
# 5. PREDICT NEW SAMPLE
# ============================================================

def predict_new_sample(text, model, kmeans_model):
    embedding = model.encode(
        [text],
        normalize_embeddings=True
    )

    cluster = kmeans_model.predict(embedding)[0]
    return cluster

# ============================================================
# 6. MAIN
# ============================================================

def main():

    INPUT_FILE = '/content/drive/MyDrive/tikharm_detection/datasets/output/hashtag_map.json'
    EMBEDDINGS_FILE = '/content/drive/MyDrive/tikharm_detection/datasets/output/kmeans_clustering/embeddings.pkl'
    OUTPUT_DIR = '/content/drive/MyDrive/tikharm_detection/datasets/output/kmeans_clustering'
    N_CLUSTERS_LIST = [20, 50, 100, 200]

    print("Loading data...")
    data = load_hashtag_data(INPUT_FILE)

    if Path(EMBEDDINGS_FILE).exists():
        embed_data = load_embeddings(EMBEDDINGS_FILE)
    else:
        embed_data = create_and_save_embeddings(data, EMBEDDINGS_FILE)

    embeddings = embed_data['embeddings']

    # KMeans
    models, cluster_results = run_kmeans_clustering(
        embeddings,
        N_CLUSTERS_LIST
    )

    # Save models
    with open(f"{OUTPUT_DIR}/kmeans_models.pkl", "wb") as f:
        pickle.dump(models, f)

    # Save results
    save_cluster_results(
        embed_data['hashtags'],
        embed_data['texts'],
        cluster_results,
        OUTPUT_DIR
    )

    # Visualization
    visualize_clusters_pca(
        embeddings,
        cluster_results,
        embed_data['texts'],
        OUTPUT_DIR
    )

    print("DONE!")

if __name__ == "__main__":
    main()

Loading data...
Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Embedding batch 1
Embedding batch 2
Embedding batch 3
Embedding batch 4
Embedding batch 5
Embedding batch 6
Embedding batch 7
Embedding batch 8
Embedding batch 9
Embedding batch 10
Embedding batch 11
Embedding batch 12
Embedding batch 13
Embedding batch 14
Embedding batch 15
Embedding batch 16
Embedding batch 17
Embedding batch 18
Embedding batch 19
Embedding batch 20
Embedding batch 21
Embedding batch 22
Embedding batch 23
Embedding batch 24
Embedding batch 25
Embedding batch 26
Embedding batch 27
Embedding batch 28
Embedding batch 29
Embedding batch 30
Embedding batch 31
Embedding batch 32
Embedding batch 33
Embedding batch 34
Embedding batch 35
Embedding batch 36
Embedding batch 37
Embedding batch 38
Embedding batch 39
Embedding batch 40
Embedding batch 41
Embedding batch 42
Embedding batch 43
Saved embeddings
Running KMeans with 20 clusters...
Running KMeans with 50 clusters...
Running KMeans with 100 clusters...
Running KMeans with 200 clusters...
DONE!


In [10]:
import pandas as pd
from collections import defaultdict


# ==========================================
# 1. LOAD HASHTAG DISTRIBUTION
# ==========================================

def load_hashtag_distribution(txt_file):

    hashtag_label_freq = defaultdict(lambda: defaultdict(int))

    current_hashtag = None
    reading_section = False

    with open(txt_file, "r", encoding="utf-8") as f:

        for line in f:

            line = line.rstrip()

            # bắt đầu section
            if "===== HASHTAG DISTRIBUTION ACROSS LABELS =====" in line:
                reading_section = True
                continue

            # dừng section
            if "===== HASHTAGS ONLY APPEAR IN ONE LABEL =====" in line:
                break

            if not reading_section:
                continue

            if not line.strip():
                continue

            # hashtag line
            if not line.startswith(" "):
                current_hashtag = line.strip().lower()
                continue

            # label line
            label, count = line.split(":")
            label = label.strip()
            count = int(count.strip())

            hashtag_label_freq[current_hashtag][label] += count

    return hashtag_label_freq


# ==========================================
# 2. LOAD CLUSTER CSV
# ==========================================

def load_cluster_csv(csv_file):

    df = pd.read_csv(csv_file)

    df["hashtag"] = df["hashtag"].str.replace("#", "").str.lower()

    return df


# ==========================================
# 3. COMPUTE CLUSTER LABEL RATIO
# ==========================================

def compute_cluster_ratios(df, hashtag_label_freq):

    cluster_counts = defaultdict(lambda: defaultdict(int))

    for _, row in df.iterrows():

        hashtag = row["hashtag"]
        cluster = row["cluster"]

        if hashtag not in hashtag_label_freq:
            continue

        for label, freq in hashtag_label_freq[hashtag].items():
            cluster_counts[cluster][label] += freq

    results = []

    for cluster, label_counts in cluster_counts.items():

        total = sum(label_counts.values())

        row = {"cluster": cluster}

        for label, count in label_counts.items():
            row[label] = count / total

        results.append(row)

    df_result = pd.DataFrame(results).fillna(0)

    return df_result


# ==========================================
# 4. SAVE RESULT
# ==========================================

def save_cluster_distribution(df, output_file):

    df = df.sort_values("cluster")

    df.to_csv(output_file, index=False)

    print("Saved:", output_file)


# ==========================================
# RUN
# ==========================================

cluster_csv = "/content/drive/MyDrive/tikharm_detection/datasets/output/kmeans_clustering/clusters_100.csv"
hashtag_stats_txt = "/content/drive/MyDrive/tikharm_detection/datasets/output/hashtag_statistics.txt"
output_csv = "/content/drive/MyDrive/tikharm_detection/datasets/output/kmeans_clustering/cluster_label_distribution_100.csv"

freq_data = load_hashtag_distribution(hashtag_stats_txt)

cluster_df = load_cluster_csv(cluster_csv)

result_df = compute_cluster_ratios(cluster_df, freq_data)

save_cluster_distribution(result_df, output_csv)

Saved: /content/drive/MyDrive/tikharm_detection/datasets/output/kmeans_clustering/cluster_label_distribution_100.csv


In [16]:
# meta_cluster_and_pca.py
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering, KMeans

# -----------------------------
# Config
# -----------------------------
INPUT_CSV = "/content/drive/MyDrive/tikharm_detection/datasets/output/kmeans_clustering/cluster_label_distribution_20.csv"  # file từ bước trước
N_META_CLUSTERS = 5                          # số meta-clusters mong muốn
OUTPUT_DIR = "/content/drive/MyDrive/tikharm_detection/datasets/output/kmeans_clustering/meta_cluster_output/20_clusters"
CLUSTERING_METHOD = "kmeans"           # "agglomerative" | "kmeans"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# -----------------------------
# 1) Load CSV & preprocess
# -----------------------------
def load_and_prepare(input_csv):
    df = pd.read_csv(input_csv)
    # ensure 'cluster' column present
    if 'cluster' not in df.columns:
        raise ValueError("Input CSV must contain a 'cluster' column")

    # sort by cluster for stable ordering
    df = df.sort_values('cluster').reset_index(drop=True)

    # extract features (all columns except 'cluster')
    feature_cols = [c for c in df.columns if c != 'cluster']
    X = df[feature_cols].fillna(0).astype(float).values

    # keep cluster ids for later annotation
    cluster_ids = df['cluster'].values

    return df, cluster_ids, feature_cols, X


# -----------------------------
# 2) Standardize features
# -----------------------------
def standardize_features(X):
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    return Xs, scaler


# -----------------------------
# 3) Meta-clustering
# -----------------------------
def meta_cluster(Xs, n_meta=N_META_CLUSTERS, method="agglomerative"):
    if method == "agglomerative":
        model = AgglomerativeClustering(n_clusters=n_meta,metric="euclidean",linkage="ward")
        labels = model.fit_predict(Xs)
    elif method == "kmeans":
        model = KMeans(n_clusters=n_meta, n_init=10, random_state=42)
        labels = model.fit_predict(Xs)
    else:
        raise ValueError("Unknown method: choose 'agglomerative' or 'kmeans'")

    return labels, model


# -----------------------------
# 4) PCA for visualization
# -----------------------------
def pca_2d(Xs):
    pca = PCA(n_components=2, random_state=42)
    X2 = pca.fit_transform(Xs)
    explained = pca.explained_variance_ratio_
    return X2, pca, explained


# -----------------------------
# 5) Save results
# -----------------------------
def save_results(df_orig, cluster_ids, feature_cols, meta_labels, X2, explained, outdir):

    # DataFrame with meta-cluster
    out_df = df_orig.copy()
    out_df['meta_cluster'] = meta_labels
    out_csv = Path(outdir) / "cluster_meta_mapping.csv"
    out_df.to_csv(out_csv, index=False, encoding='utf-8')
    print("Saved meta mapping:", out_csv)

    # Save PCA scatter plot
    fig, ax = plt.subplots(figsize=(12, 9))
    scatter = ax.scatter(X2[:, 0], X2[:, 1], c=meta_labels, cmap='tab20', s=120, alpha=0.8)
    ax.set_title(f"PCA (2D) of clusters — meta-clusters: {len(np.unique(meta_labels))}\n"
                 f"Explained var: PC1 {explained[0]:.2%}, PC2 {explained[1]:.2%}")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

    # annotate cluster ids (optional: annotate only if few points)
    # annotate only first N if too many
    annotate = True
    max_annotate = 200
    if X2.shape[0] <= max_annotate and annotate:
        for i, cid in enumerate(cluster_ids):
            ax.annotate(str(cid), (X2[i, 0], X2[i, 1]), fontsize=8, alpha=0.9)

    plt.colorbar(scatter, ax=ax, label='meta-cluster')
    plt.tight_layout()
    fig_path = Path(outdir) / "pca_meta_clusters.png"
    plt.savefig(fig_path, dpi=200)
    plt.close()
    print("Saved PCA plot:", fig_path)

    # Also save a summary per meta-cluster: mean proportions and counts
    summary_rows = []
    for m in sorted(np.unique(meta_labels)):
        mask = out_df['meta_cluster'] == m
        subset = out_df.loc[mask, feature_cols]
        mean_props = subset.mean().to_dict()
        count = mask.sum()
        row = {'meta_cluster': m, 'count': int(count)}
        row.update(mean_props)
        summary_rows.append(row)
    summary_df = pd.DataFrame(summary_rows).fillna(0)
    summary_csv = Path(outdir) / "meta_cluster_summary.csv"
    summary_df.to_csv(summary_csv, index=False, encoding='utf-8')
    print("Saved meta-cluster summary:", summary_csv)


# -----------------------------
# Main flow
# -----------------------------
def main():
    df, cluster_ids, feature_cols, X = load_and_prepare(INPUT_CSV)
    Xs, scaler = standardize_features(X)
    meta_labels, model = meta_cluster(Xs, n_meta=N_META_CLUSTERS, method=CLUSTERING_METHOD)
    X2, pca, explained = pca_2d(Xs)
    save_results(df, cluster_ids, feature_cols, meta_labels, X2, explained, OUTPUT_DIR)
    print("Done.")


if __name__ == "__main__":
    main()

Saved meta mapping: /content/drive/MyDrive/tikharm_detection/datasets/output/kmeans_clustering/meta_cluster_output/20_clusters/cluster_meta_mapping.csv
Saved PCA plot: /content/drive/MyDrive/tikharm_detection/datasets/output/kmeans_clustering/meta_cluster_output/20_clusters/pca_meta_clusters.png
Saved meta-cluster summary: /content/drive/MyDrive/tikharm_detection/datasets/output/kmeans_clustering/meta_cluster_output/20_clusters/meta_cluster_summary.csv
Done.


## Inference

In [20]:
model = SentenceTransformer('BAAI/bge-m3')

with open("/content/drive/MyDrive/tikharm_detection/datasets/output/kmeans_clustering/kmeans_models.pkl", "rb") as f:
    models = pickle.load(f)

kmeans = models[50]  # chọn số cluster bạn muốn

new_text = "toys"

cluster = predict_new_sample(new_text, model, kmeans)

print("Cluster:", cluster)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Cluster: 23
